In [1]:
''' Imports '''

import pandas as pd
import polars as pl
import numpy as np
import math
from time import time
from datetime import timedelta

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from scipy.stats.mstats import trimmed_var
from scipy.stats import percentileofscore

from sklearn.cluster import KMeans, AffinityPropagation, MeanShift, SpectralClustering, DBSCAN, HDBSCAN, OPTICS, estimate_bandwidth
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.preprocessing import StandardScaler, Normalizer

from prep_data import load_pbp_participation_data, load_stats_team_tendencies_offense, load_stats_team_tendencies_defense

# Get Data

In [3]:
offense_tendencies = load_stats_team_tendencies_offense()

print(offense_tendencies.head().to_string())

                Games  Drives  Plays  Neutral_Down_Plays  Pass_Plays  Neutral_Down_Pass  Pass_Attempts  QBScrambles     IAY  IAY_ToSticks  TotalTimeToThrow  Pass_BehindLOS  Pass_Short  Pass_Medium  Pass_Deep  Sacks  Rush_Plays  Rush_Attempts  Rush_Inside  Rush_Outside  Plays_11_Personnel  Plays_Heavy_Personnel  Plays_Mult_RBs  Plays_Zero_RBs  Plays_Mult_TEs  Plays_Zero_TEs  Plays_Extra_OL  Plays / Game  Drives / Game    % Pass  % Pass Neutral Downs  Scrambles / Game      ADOT  ADOT to Sticks  Avg Time to Throw  % Passes Behind LOS  % Passes Short  % Passes Medium  % Passes Deep  % Rush Inside  % Rush Outside  % Plays 11 Personnel  % Plays Heavy Personnel  % Plays Mult RBs  % Plays Zero RBs  % Plays Mult TEs  % Plays Zero TEs  % Plays Extra OL  Shotgun Plays  Under Center Plays  Shotgun Neutral_Down_Plays  Under Center Neutral_Down_Plays  Shotgun % Pass  Under Center % Pass  % Under Center  % Shotgun  % Under Center Neutral Downs  % Shotgun Neutral Downs  MaxTargets  MaxTargetShare  N_R

In [4]:
''' Features '''


FEATURES = [
    'Plays / Game', 'Drives / Game', 'Scrambles / Game', 
    '% Plays 11 Personnel', '% Under Center', 'Shotgun % Pass', 'Under Center % Pass',
    'ADOT', 'Avg Time to Throw', '% Passes Behind LOS', '% Passes Deep', 'MaxTargetShare',
    '% Rush Outside', 'MaxRushAttemptsShare',
]

VIZ_FEATURES = ['Plays / Game', '% Pass', 'Scrambles / Game', 
                '% Plays Plays 11 Personnel',
                '% Under Center', 'ADOT', 'Avg Time to Throw', 'MaxTargetShare', 
                '% Rush Outside', 'MaxRushAttemptsShare']

# Preprocessing

In [5]:
''' Transform and Scale '''

# Log transform data
# transformed_data = pd.DataFrame(np.log(offense_tendencies[OFFENSE_FEATURES]), columns=OFFENSE_FEATURES).replace(math.inf, 0).replace(-(math.inf), 0)

# Scale data
scaler = StandardScaler()
scaled_data = scaler.fit_transform(offense_tendencies[FEATURES])
scaled_data_df = pd.DataFrame(scaled_data, columns=FEATURES)

print(scaled_data_df.shape)
print(scaled_data_df.head().to_string())

# Scale data
norm = Normalizer()
norm_data = norm.fit_transform(offense_tendencies[FEATURES])
norm_data_df = pd.DataFrame(norm_data, columns=FEATURES)

print(norm_data_df.shape)
print(norm_data_df.head().to_string())

(288, 14)
   Plays / Game  Drives / Game  Scrambles / Game  % Plays 11 Personnel  % Under Center  Shotgun % Pass  Under Center % Pass      ADOT  Avg Time to Throw  % Passes Behind LOS  % Passes Deep  MaxTargetShare  % Rush Outside  MaxRushAttemptsShare
0      0.753261       1.547027         -1.524084             -0.128782        1.423627        1.427770             0.417428  2.060821          -1.494379            -2.297630       0.119912        0.508965       -0.528517              1.955732
1      0.811265       1.547027         -0.815344             -0.678900        1.751401        2.027936             0.878225  1.694859          -1.087927            -1.056597      -0.261757        1.331324       -1.367631             -1.300776
2     -2.378942       2.039292         -0.714096              0.444104        0.977525        0.947777             0.319710  0.354440          -0.019875             0.401781       0.203802        0.560056       -1.963273              1.691972
3      0.811265   

In [16]:
''' t-SNE '''

# Final
PERPLEXITY = 30
TSNE_N_COMPONENTS = 3
TSNE_N_COMPONENT_NAMES = [f'TSNE Component {n+1}' for n in range(TSNE_N_COMPONENTS)]

# Model
tsne_model = TSNE(
    n_components=TSNE_N_COMPONENTS, 
    perplexity=PERPLEXITY, 
    random_state=42, 
    init='pca'
)

# Fit
y = tsne_model.fit_transform(offense_tendencies[FEATURES])
print(f'Divergence:', tsne_model.kl_divergence_)

# Results df
tsne_df = pd.DataFrame(y, columns=TSNE_N_COMPONENT_NAMES)

print(tsne_df.shape)
print(tsne_df.head().to_string())

Divergence: 0.3623304069042206
(288, 3)
   TSNE Component 1  TSNE Component 2  TSNE Component 3
0          4.508006         -3.830941          0.035581
1          4.966401         -3.647205          0.112400
2        -10.204571         -2.026768          1.077301
3          5.486273          1.583986         -0.829240
4          7.279263          0.062538          1.345048


In [17]:
def tsne_chart(colors: list = None):

    fig = go.Figure()
    if TSNE_N_COMPONENTS == 2:
        fig = px.scatter(
            x=tsne_df['TSNE Component 1'],
            y=tsne_df['TSNE Component 2'],
            color=colors,
        )
    else:
        fig = px.scatter_3d(
            x=tsne_df['TSNE Component 1'],
            y=tsne_df['TSNE Component 2'],
            z=tsne_df['TSNE Component 3'],
            color=colors,
        )
    fig.update_layout(
        xaxis_title='TSNE Component 1', 
        yaxis_title='TSNE Component 2'
    )
        
    return fig


fig = tsne_chart()
fig.show()

# Feature Selection

1. Raw Features
    1. Could do Pearson feature selection variant; instead of features with highest correlation to Y, features with most variance, then remove features with collinearity > some threshold
1. PCA

In [ ]:
''' PCA - final '''

# Set number of PCA components to use after initial try
PCA_N_COMPONENTS = 8
PCA_COMPONENT_COLS = [f'PCA Component {n}' for n in range(1, PCA_N_COMPONENTS + 1)]

# Instantiate transformer
pca_final = PCA(n_components=PCA_N_COMPONENTS, random_state=42)

# Transform sku profiles
pca_component_data_final = pca_final.fit_transform(scaled_data_df)

# Evaluate components
total_variance = scaled_data_df.var().sum()
expl_variance = pca_final.explained_variance_.sum()

print(f'Data set variance: {total_variance:,.3f}')
print(f'PCA explained variance: {expl_variance:,.3f} ({round((expl_variance / total_variance) * 100, 2)}%)')

pcs = pd.DataFrame(pca_final.components_, columns=FEATURES)

# Create bar charts of contribution
for n in range(2):      #PCA_N_COMPONENTS):
    pc = pcs.transpose()[n].sort_values(ascending=False)

    fig = px.bar(
        x=pc,
        y=pc.index,
        title=f"PC{n+1}: Greatest contributors"
    )
    fig.update_layout(
        xaxis_title="Correlation", 
        yaxis_title="Features", 
        yaxis={'dtick': 1, 'categoryorder':'total ascending'},
    )
    fig.show()

    comp_expl_variance = pca_final.explained_variance_[n]
    print(f'Explained variance: {comp_expl_variance:,} ({round((comp_expl_variance / total_variance) * 100, 2)}%)')
                                                                                                                                                                                                              
# Make df of PCA scores
pca_component_df = pd.DataFrame(data=pca_component_data_final, columns=PCA_COMPONENT_COLS)
print(pca_component_df.shape)
print(pca_component_df.head().to_string())

# Add PCA scores to original dataframe
offense_tendencies = offense_tendencies.drop(columns=list(filter(lambda x: x.startswith("Component"), offense_tendencies.columns)))
offense_tendencies = offense_tendencies.reset_index().merge(pca_component_df, left_index=True, right_index=True, how='left').set_index(['posteam', 'season'])


print(f'PCA values')
print(offense_tendencies.head().to_string())

# Clustering Experiment

https://scikit-learn.org/stable/modules/clustering.html  
Geometry *appears* to be spherical, which would mean non-flat. Which would lend to:
1. Affinity Propagation
1. Mean-shift
1. Spectral Clustering
1. DBSCAN
1. HDBSCAN
1. OPTICS

Not sure on number of clusters (few vs. many). I don't see many obvious clusters based on PCA / t-SNE visualization

The last 3 are density based and suited to very large N samples, which isn't this

In [18]:
''' Input '''

# Input DF
cluster_input = norm_data_df.copy()

# Constants
cluster_n_features = len(cluster_input.columns)
teams = [f'{i[0]} {i[1]}' for i in offense_tendencies.index]

print(teams)
print(f'{cluster_n_features = }')
print(cluster_input.head().to_string())

['ARI 2016', 'ARI 2017', 'ARI 2018', 'ARI 2019', 'ARI 2020', 'ARI 2021', 'ARI 2022', 'ARI 2023', 'ARI 2024', 'ATL 2016', 'ATL 2017', 'ATL 2018', 'ATL 2019', 'ATL 2020', 'ATL 2021', 'ATL 2022', 'ATL 2023', 'ATL 2024', 'BAL 2016', 'BAL 2017', 'BAL 2018', 'BAL 2019', 'BAL 2020', 'BAL 2021', 'BAL 2022', 'BAL 2023', 'BAL 2024', 'BUF 2016', 'BUF 2017', 'BUF 2018', 'BUF 2019', 'BUF 2020', 'BUF 2021', 'BUF 2022', 'BUF 2023', 'BUF 2024', 'CAR 2016', 'CAR 2017', 'CAR 2018', 'CAR 2019', 'CAR 2020', 'CAR 2021', 'CAR 2022', 'CAR 2023', 'CAR 2024', 'CHI 2016', 'CHI 2017', 'CHI 2018', 'CHI 2019', 'CHI 2020', 'CHI 2021', 'CHI 2022', 'CHI 2023', 'CHI 2024', 'CIN 2016', 'CIN 2017', 'CIN 2018', 'CIN 2019', 'CIN 2020', 'CIN 2021', 'CIN 2022', 'CIN 2023', 'CIN 2024', 'CLE 2016', 'CLE 2017', 'CLE 2018', 'CLE 2019', 'CLE 2020', 'CLE 2021', 'CLE 2022', 'CLE 2023', 'CLE 2024', 'DAL 2016', 'DAL 2017', 'DAL 2018', 'DAL 2019', 'DAL 2020', 'DAL 2021', 'DAL 2022', 'DAL 2023', 'DAL 2024', 'DEN 2016', 'DEN 2017', 'DE

In [19]:
''' Nearest Neighbors '''
## GOAL: estimate ideal parameter values for epsilon (eps)

# Init model
k_neighbors_model = NearestNeighbors(n_neighbors=cluster_n_features)

# Fit and find nearest neighbors
k_neighbors_model.fit(cluster_input)
distances, indices = k_neighbors_model.kneighbors(cluster_input)

# Plot sorted distances, find the "elbow"
distances = np.sort(distances, axis=0)
distances = distances[:,1]

fig = px.line(
    data_frame=distances
)
fig.update_layout(xaxis_title='N Neighbors', yaxis_title='Distances (EPS)')
fig.show()

In [20]:
''' Experiment '''
# https://scikit-learn.org/stable/auto_examples/cluster/plot_cluster_comparison.html


## Parameters

params = {
    "quantile": 0.3,
    "eps": .01,
    "damping": 0.9,
    "preference": -200,
    "n_neighbors": 3,
    "n_clusters": 3,
    "min_samples": 7,
    "xi": 0.05,
    "min_cluster_size": 0.1,
    "allow_single_cluster": True,
    "hdbscan_min_cluster_size": 15,
    "hdbscan_min_samples": 3,
    "random_state": 42,
}

# estimate bandwidth for mean shift
bandwidth = estimate_bandwidth(cluster_input, quantile=params["quantile"])

## Models ##

affinity_propagation = AffinityPropagation(
    damping=params["damping"],
    preference=params["preference"],
    random_state=params["random_state"],
)

ms = MeanShift(bandwidth=bandwidth, bin_seeding=True)

spectral = SpectralClustering(
    n_clusters=params["n_clusters"],
    eigen_solver="arpack",
    affinity="nearest_neighbors",
    random_state=params["random_state"],
)
dbscan = DBSCAN(eps=params["eps"])

hdbscan = HDBSCAN(
    min_samples=params["hdbscan_min_samples"],
    min_cluster_size=params["hdbscan_min_cluster_size"],
    allow_single_cluster=params["allow_single_cluster"],
    copy=True,
)

optics = OPTICS(
    min_samples=params["min_samples"],
    xi=params["xi"],
    min_cluster_size=params["min_cluster_size"],
)

clustering_algorithms = (
    ("Affinity\nPropagation", affinity_propagation),
    ("MeanShift", ms),
    ("Spectral\nClustering", spectral),
    ("DBSCAN", dbscan),
    ("HDBSCAN", hdbscan),
    ("OPTICS", optics)
)

## Evaluate ##

for name, algorithm in clustering_algorithms:
    print(name)

    # Fit
    t0 = time()
    algorithm.fit(cluster_input)
    t1 = time()

    # Labels
    y_pred = None
    if hasattr(algorithm, "labels_"):
        y_pred = algorithm.labels_.astype(int)
    else:
        y_pred = algorithm.predict(cluster_input)
    
    n_clusters_ = len(set(y_pred)) - (1 if -1 in y_pred else 0)

    # Visualize
    fig = tsne_chart(colors=y_pred.astype(str))
    fig.update_layout(
        title=f'{name}<br><sup>{n_clusters_ = }</sup>',
    )
    fig.show()

Affinity
Propagation


MeanShift


Spectral
Clustering


DBSCAN


HDBSCAN


OPTICS


# Spectral

In [11]:
spectral_n_clusters = [4, 6, 8, 10]

for n_clusters in spectral_n_clusters:
    # Model
    spectral = SpectralClustering(
        n_clusters=n_clusters,
        eigen_solver="arpack",
        affinity="nearest_neighbors",
        random_state=42,
    )
    
    # Fit
    spectral.fit(cluster_input)

    # Labels
    labels = spectral.labels_.astype(str)

    # Visualize
    fig = tsne_chart(colors=labels)
    fig.update_layout(
        title=f'Spectral Clustering<br><sup>{n_clusters = }</sup>'
    )
    fig.show()

# HDBSCAN

In [ ]:
''' Experiment '''

min_cluster_size_options = [2, 3, 5, 7, 9]
min_samples_options = [2, 3, 5, 7, 9]
cluster_selection_method = 'leaf'

for min_cluster_size in min_cluster_size_options:
    for min_samples in min_samples_options:
        # Create model
        cluster_model = HDBSCAN(
            copy=True,
            min_cluster_size=min_cluster_size,
            min_samples=min_samples,
            cluster_selection_method=cluster_selection_method
        )

        # Fit
        cluster_model.fit(cluster_input)

        # Number of clusters, ignoring noise
        labels = cluster_model.labels_
        n_clusters_ = len(set(labels)) - (1 if -1 in labels else 0)
        n_noise_ = list(labels).count(-1)

        # Visualize
        fig = tsne_chart(labels.astype(str))
        fig.update_layout(
            title=f'HDBSCAN<br><sup>{n_clusters_ = } | {n_noise_ = } | {min_cluster_size = } | {min_samples = } | {cluster_selection_method = }</sup>'
        )
        fig.show()
